In [9]:
# Outcome Model 01 v1: Plain logistic regression, no winsorization
# Thesis-standard implementation under final outcome v1 six-fold expanding-window protocol
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Outcome_Master_Modeling_Panel_v1.csv"

STAGE_METRICS_CSV = BASE_DIR / "Outcome_Model_01_Logit_NoWinsor_v1_Stage_Metrics.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Outcome_Model_01_Logit_NoWinsor_v1_DevCV_Fold_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Outcome_Model_01_Logit_NoWinsor_v1_Predictions.csv"
COEFFICIENTS_CSV = BASE_DIR / "Outcome_Model_01_Logit_NoWinsor_v1_Coefficients.csv"

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. PREPROCESSING PIPELINE
# =========================================================
# No winsorization in Model 01.
# Continuous features: median impute + scale
# Binary features: fill missing with 0, then pass through
# FF17 categorical control: fixed one-hot with "Other" dropped as reference

continuous_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

binary_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
        (
            "onehot",
            OneHotEncoder(
                categories=[ff17_categories],
                drop=["Other"],           # locked reference category
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cont", continuous_transformer, continuous_features),
        ("bin", binary_transformer, binary_features),
        ("ff17", categorical_transformer, [categorical_feature]),
    ],
    remainder="drop",
)

# Plain logistic, no class weighting, no oversampling, no winsorization
# If your sklearn version errors on penalty=None, replace with penalty="none".
logit_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logit_model),
    ]
)

# =========================================================
# 7. OPTION B: DEVELOPMENT CV FOLDS
# =========================================================
# Expanding-window development CV under the final v2 protocol
# Fold 1: train 2010-2015 -> validate 2016
# Fold 2: train 2010-2016 -> validate 2017
# Fold 3: train 2010-2017 -> validate 2018
# Fold 4: train 2010-2018 -> validate 2019
# Fold 5: train 2010-2019 -> validate 2020
# Fold 6: train 2010-2020 -> validate 2021

cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

cv_metrics = []
prediction_frames = []

all_predictors = continuous_features + binary_features + [categorical_feature]

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline.fit(X_train, y_train)

    model = pipeline.named_steps["model"]
    print(f"[{fold_name}] model.classes_ =", model.classes_)
    assert list(model.classes_) == [0, 1], f"Unexpected class ordering: {model.classes_}"
    
    class_list = list(model.classes_)
    positive_index = class_list.index(1)
    val_prob = pipeline.predict_proba(X_val)[:, positive_index]
    
    print(f"[{fold_name}] mean prob | y=1 =", float(np.mean(val_prob[y_val.to_numpy() == 1])))
    print(f"[{fold_name}] mean prob | y=0 =", float(np.mean(val_prob[y_val.to_numpy() == 0])))

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_metrics.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_logit_nowinsor": val_prob,
            }
        )
    )
cv_metrics_df = pd.DataFrame(cv_metrics)

cv_summary = (
    cv_metrics_df[["pr_auc", "roc_auc", "brier_score"]]
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

# =========================================================
# 8. FIT ON FULL DEVELOPMENT, EVALUATE FINAL TEST
# =========================================================
development_df = df[df["split"] == "development"].copy()
final_test_df = df[df["split"] == "final_test"].copy()

X_dev = development_df[all_predictors].copy()
y_dev = development_df[TARGET_COL].copy()

pipeline.fit(X_dev, y_dev)

model = pipeline.named_steps["model"]
print("[final_test] model.classes_ =", model.classes_)
assert list(model.classes_) == [0, 1], f"Unexpected class ordering: {model.classes_}"

X_final = final_test_df[all_predictors].copy()
y_final = final_test_df[TARGET_COL].copy()

class_list = list(model.classes_)
positive_index = class_list.index(1)
final_prob = pipeline.predict_proba(X_final)[:, positive_index]

print("[final_test] mean prob | y=1 =", float(np.mean(final_prob[y_final.to_numpy() == 1])))
print("[final_test] mean prob | y=0 =", float(np.mean(final_prob[y_final.to_numpy() == 0])))

final_metrics = evaluate_predictions(y_final, final_prob)
final_metrics.update(
    {
        "stage": "final_test",
        "rows": len(final_test_df),
        "positives": int(y_final.sum()),
        "positive_rate": float(y_final.mean()),
        "mean_predicted_probability": float(np.mean(final_prob)),
    }
)

prediction_frames.append(
    pd.DataFrame(
    {
        "deal_number": final_test_df["deal_number"].to_numpy(),
        "permno": final_test_df["permno"].to_numpy(),
        "year": final_test_df["year"].to_numpy(),
        "evaluation_stage": "final_test",
        "fold": "",
            
        "outcome_dissident_favorable": y_final.to_numpy(),
        "outcome_mgmt_favorable": final_test_df["outcome_mgmt_favorable"].to_numpy(),
        "outcome_class": final_test_df["outcome_class"].to_numpy(),
        "pred_outcome_logit_nowinsor": final_prob,
    }
)
)
# =========================================================
# 9. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "pr_auc": cv_metrics_df["pr_auc"].mean(),
            "roc_auc": cv_metrics_df["roc_auc"].mean(),
            "brier_score": cv_metrics_df["brier_score"].mean(),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "pr_auc": cv_metrics_df["pr_auc"].std(ddof=1),
            "roc_auc": cv_metrics_df["roc_auc"].std(ddof=1),
            "brier_score": cv_metrics_df["brier_score"].std(ddof=1),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        final_metrics,
    ]
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)

# =========================================================
# 10. EXTRACT COEFFICIENTS FROM FULL-DEVELOPMENT FIT
# =========================================================
preprocessor_fitted = pipeline.named_steps["preprocessor"]
model_fitted = pipeline.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()
coef_values = model_fitted.coef_.ravel()

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coef_values,
        "abs_coefficient": np.abs(coef_values),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

intercept_df = pd.DataFrame(
    {
        "feature": ["INTERCEPT"],
        "coefficient": [model_fitted.intercept_[0]],
        "abs_coefficient": [abs(model_fitted.intercept_[0])],
    }
)

coef_df = pd.concat([intercept_df, coef_df], ignore_index=True)

# =========================================================
# 11. SAVE OUTPUTS
# =========================================================
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)
cv_metrics_df.to_csv(CV_FOLD_METRICS_CSV, index=False)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
coef_df.to_csv(COEFFICIENTS_CSV, index=False)

# =========================================================
# 12. DISPLAY RESULTS
# =========================================================
print("Saved:")
print(STAGE_METRICS_CSV)
print(CV_FOLD_METRICS_CSV)
print(PREDICTIONS_CSV)
print(COEFFICIENTS_CSV)

print("\nStage metrics:")
print(stage_metrics_df.to_string(index=False))

print("\nDevelopment CV fold metrics:")
print(cv_metrics_df.to_string(index=False))

print("\nTop 20 coefficients by absolute magnitude:")
print(coef_df.head(20).to_string(index=False))

[fold_1] model.classes_ = [0 1]
[fold_1] mean prob | y=1 = 0.2153071891815819
[fold_1] mean prob | y=0 = 0.24137469263252098
[fold_2] model.classes_ = [0 1]
[fold_2] mean prob | y=1 = 0.20288877354732682
[fold_2] mean prob | y=0 = 0.19088885492012353
[fold_3] model.classes_ = [0 1]
[fold_3] mean prob | y=1 = 0.10809428822487081
[fold_3] mean prob | y=0 = 0.18234251809026283
[fold_4] model.classes_ = [0 1]
[fold_4] mean prob | y=1 = 0.232444042505051
[fold_4] mean prob | y=0 = 0.1409841072438901
[fold_5] model.classes_ = [0 1]
[fold_5] mean prob | y=1 = 0.1253512068009198
[fold_5] mean prob | y=0 = 0.18050945596552925
[fold_6] model.classes_ = [0 1]
[fold_6] mean prob | y=1 = 0.13808314615168876
[fold_6] mean prob | y=0 = 0.19211418586158135
[final_test] model.classes_ = [0 1]
[final_test] mean prob | y=1 = 0.19530570279954185
[final_test] mean prob | y=0 = 0.22012559112610122
Saved:
Outcome_Model_01_Logit_NoWinsor_v1_Stage_Metrics.csv
Outcome_Model_01_Logit_NoWinsor_v1_DevCV_Fold_Metri

In [4]:
# Outcome Model 02 v1: Plain logistic regression, with winsorization
# Thesis-standard implementation under final outcome v1 six-fold expanding-window protocol
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Outcome_Master_Modeling_Panel_v1.csv"

STAGE_METRICS_CSV = BASE_DIR / "Outcome_Model_02_Logit_Winsor_v1_Stage_Metrics.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Outcome_Model_02_Logit_Winsor_v1_DevCV_Fold_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Outcome_Model_02_Logit_Winsor_v1_Predictions.csv"
COEFFICIENTS_CSV = BASE_DIR / "Outcome_Model_02_Logit_Winsor_v1_Coefficients.csv"

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSING PIPELINE
# =========================================================
# Continuous: winsorize -> median impute -> scale
# Binary: fill missing with 0
# FF17: fixed one-hot with "Other" dropped as reference

continuous_transformer = Pipeline(
    steps=[
        ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

binary_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
        (
            "onehot",
            OneHotEncoder(
                categories=[ff17_categories],
                drop=["Other"],
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cont", continuous_transformer, continuous_features),
        ("bin", binary_transformer, binary_features),
        ("ff17", categorical_transformer, [categorical_feature]),
    ],
    remainder="drop",
)

# Plain logistic, no class weighting, no oversampling
logit_model = LogisticRegression(
    solver="lbfgs",
    max_iter=5000,
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logit_model),
    ]
)

# =========================================================
# 8. OPTION B: DEVELOPMENT CV FOLDS
# =========================================================
# Expanding-window development CV under the final v2 protocol
# Fold 1: train 2010-2015 -> validate 2016
# Fold 2: train 2010-2016 -> validate 2017
# Fold 3: train 2010-2017 -> validate 2018
# Fold 4: train 2010-2018 -> validate 2019
# Fold 5: train 2010-2019 -> validate 2020
# Fold 6: train 2010-2020 -> validate 2021

cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

cv_metrics = []
prediction_frames = []

all_predictors = continuous_features + binary_features + [categorical_feature]

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_metrics.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_logit_winsor": val_prob,
            }
        )
    )
cv_metrics_df = pd.DataFrame(cv_metrics)

# =========================================================
# 9. FIT ON FULL DEVELOPMENT, EVALUATE FINAL TEST
# =========================================================
development_df = df[df["split"] == "development"].copy()
final_test_df = df[df["split"] == "final_test"].copy()

X_dev = development_df[all_predictors].copy()
y_dev = development_df[TARGET_COL].copy()

pipeline.fit(X_dev, y_dev)

# Final test
X_final = final_test_df[all_predictors].copy()
y_final = final_test_df[TARGET_COL].copy()
final_prob = pipeline.predict_proba(X_final)[:, 1]

final_metrics = evaluate_predictions(y_final, final_prob)
final_metrics.update(
    {
        "stage": "final_test",
        "rows": len(final_test_df),
        "positives": int(y_final.sum()),
        "positive_rate": float(y_final.mean()),
        "mean_predicted_probability": float(np.mean(final_prob)),
    }
)

prediction_frames.append(
    pd.DataFrame(
    {
        "deal_number": final_test_df["deal_number"].to_numpy(),
        "permno": final_test_df["permno"].to_numpy(),
        "year": final_test_df["year"].to_numpy(),
        "evaluation_stage": "final_test",
        "fold": "",
            
        "outcome_dissident_favorable": y_final.to_numpy(),
        "outcome_mgmt_favorable": final_test_df["outcome_mgmt_favorable"].to_numpy(),
        "outcome_class": final_test_df["outcome_class"].to_numpy(),
        "pred_outcome_logit_winsor": final_prob,
    }
)
)
# =========================================================
# 10. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "pr_auc": cv_metrics_df["pr_auc"].mean(),
            "roc_auc": cv_metrics_df["roc_auc"].mean(),
            "brier_score": cv_metrics_df["brier_score"].mean(),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "pr_auc": cv_metrics_df["pr_auc"].std(ddof=1),
            "roc_auc": cv_metrics_df["roc_auc"].std(ddof=1),
            "brier_score": cv_metrics_df["brier_score"].std(ddof=1),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        final_metrics,
    ]
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)

# =========================================================
# 11. EXTRACT COEFFICIENTS FROM FULL-DEVELOPMENT FIT
# =========================================================
preprocessor_fitted = pipeline.named_steps["preprocessor"]
model_fitted = pipeline.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()
coef_values = model_fitted.coef_.ravel()

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coef_values,
        "abs_coefficient": np.abs(coef_values),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

intercept_df = pd.DataFrame(
    {
        "feature": ["INTERCEPT"],
        "coefficient": [model_fitted.intercept_[0]],
        "abs_coefficient": [abs(model_fitted.intercept_[0])],
    }
)

coef_df = pd.concat([intercept_df, coef_df], ignore_index=True)

# =========================================================
# 12. SAVE OUTPUTS
# =========================================================
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)
cv_metrics_df.to_csv(CV_FOLD_METRICS_CSV, index=False)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
coef_df.to_csv(COEFFICIENTS_CSV, index=False)

# =========================================================
# 13. DISPLAY RESULTS
# =========================================================
print("Saved:")
print(STAGE_METRICS_CSV)
print(CV_FOLD_METRICS_CSV)
print(PREDICTIONS_CSV)
print(COEFFICIENTS_CSV)

print("\nStage metrics:")
print(stage_metrics_df.to_string(index=False))

print("\nDevelopment CV fold metrics:")
print(cv_metrics_df.to_string(index=False))

print("\nTop 20 coefficients by absolute magnitude:")
print(coef_df.head(20).to_string(index=False))

Saved:
Outcome_Model_02_Logit_Winsor_v1_Stage_Metrics.csv
Outcome_Model_02_Logit_Winsor_v1_DevCV_Fold_Metrics.csv
Outcome_Model_02_Logit_Winsor_v1_Predictions.csv
Outcome_Model_02_Logit_Winsor_v1_Coefficients.csv

Stage metrics:
              stage   pr_auc  roc_auc  brier_score  rows  positives  positive_rate  mean_predicted_probability
development_cv_mean 0.228079 0.460129     0.182436   NaN        NaN            NaN                         NaN
 development_cv_std 0.080580 0.058597     0.043602   NaN        NaN            NaN                         NaN
         final_test 0.253839 0.400747     0.225410 233.0       63.0       0.270386                    0.207013

Development CV fold metrics:
  pr_auc  roc_auc  brier_score   fold  train_year_start  train_year_end  val_year_start  val_year_end  train_rows  train_positives  train_positive_rate  val_rows  val_positives  val_positive_rate  mean_predicted_probability
0.172130 0.414944     0.209697 fold_1              2010            2015  

In [5]:
# Outcome Model 03 v1: Ridge logistic regression, with winsorization
# C tuned by six-fold expanding-window CV through 2021
# Thesis-standard implementation under final outcome v1 split protocol
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Outcome_Master_Modeling_Panel_v1.csv"

TUNING_SUMMARY_CSV = BASE_DIR / "Outcome_Model_03_Ridge_Winsor_v1_Tuning_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Outcome_Model_03_Ridge_Winsor_v1_SelectedC_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Outcome_Model_03_Ridge_Winsor_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Outcome_Model_03_Ridge_Winsor_v1_Predictions.csv"
COEFFICIENTS_CSV = BASE_DIR / "Outcome_Model_03_Ridge_Winsor_v1_Coefficients.csv"

# Candidate ridge strengths
C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSING + MODEL BUILDER
# =========================================================
def build_pipeline(C_value: float) -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[
            ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    ridge_model = LogisticRegression(
        C=C_value,
        solver="lbfgs",
        max_iter=5000,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", ridge_model),
        ]
    )

# =========================================================
# 8. OPTION B: DEVELOPMENT CV FOLDS
# =========================================================
# Expanding-window development CV under the final v2 protocol
# Fold 1: train 2010-2015 -> validate 2016
# Fold 2: train 2010-2016 -> validate 2017
# Fold 3: train 2010-2017 -> validate 2018
# Fold 4: train 2010-2018 -> validate 2019
# Fold 5: train 2010-2019 -> validate 2020
# Fold 6: train 2010-2020 -> validate 2021

cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]


# =========================================================
# 9. TUNE C BY DEVELOPMENT CV
# =========================================================
tuning_rows = []

for C_value in C_GRID:
    for fold_def in cv_fold_defs:
        fold_name = fold_def["fold"]
        train_years = fold_def["train_years"]
        val_years = fold_def["val_years"]

        train_df = df[df["year"].isin(train_years)].copy()
        val_df = df[df["year"].isin(val_years)].copy()

        X_train = train_df[all_predictors].copy()
        y_train = train_df[TARGET_COL].copy()

        X_val = val_df[all_predictors].copy()
        y_val = val_df[TARGET_COL].copy()

        pipeline = build_pipeline(C_value)
        pipeline.fit(X_train, y_train)
        val_prob = pipeline.predict_proba(X_val)[:, 1]

        fold_metrics = evaluate_predictions(y_val, val_prob)
        fold_metrics.update(
            {
                "C": C_value,
                "fold": fold_name,
                "train_year_start": min(train_years),
                "train_year_end": max(train_years),
                "val_year_start": min(val_years),
                "val_year_end": max(val_years),
                "train_rows": len(train_df),
                "train_positives": int(y_train.sum()),
                "train_positive_rate": float(y_train.mean()),
                "val_rows": len(val_df),
                "val_positives": int(y_val.sum()),
                "val_positive_rate": float(y_val.mean()),
                "mean_predicted_probability": float(np.mean(val_prob)),
            }
        )
        tuning_rows.append(fold_metrics)

tuning_detail_df = pd.DataFrame(tuning_rows)

tuning_summary_df = (
    tuning_detail_df
    .groupby("C", as_index=False)
    .agg(
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_brier_score=("brier_score", "mean"),
        std_brier_score=("brier_score", "std"),
    )
)

# Selection rule:
# 1. highest mean PR-AUC
# 2. if tie, highest mean ROC-AUC
# 3. if tie, smaller C (more conservative regularization)
tuning_summary_df = tuning_summary_df.sort_values(
    by=["mean_pr_auc", "mean_roc_auc", "C"],
    ascending=[False, False, True]
).reset_index(drop=True)

selected_C = float(tuning_summary_df.loc[0, "C"])

# =========================================================
# 10. RERUN SELECTED C FOR FOLD METRICS + OOS PREDICTIONS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(selected_C)
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "selected_C": selected_C,
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "selected_C": selected_C,
                
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_ridge_winsor": val_prob,
            }
        )
    )
selected_cv_metrics_df = pd.DataFrame(selected_cv_rows)

# =========================================================
# 11. FIT SELECTED C ON FULL DEVELOPMENT, EVALUATE FINAL TEST
# =========================================================
development_df = df[df["split"] == "development"].copy()
final_test_df = df[df["split"] == "final_test"].copy()

X_dev = development_df[all_predictors].copy()
y_dev = development_df[TARGET_COL].copy()

final_pipeline = build_pipeline(selected_C)
final_pipeline.fit(X_dev, y_dev)

# Final test
X_final = final_test_df[all_predictors].copy()
y_final = final_test_df[TARGET_COL].copy()
final_prob = final_pipeline.predict_proba(X_final)[:, 1]

final_metrics = evaluate_predictions(y_final, final_prob)
final_metrics.update(
    {
        "stage": "final_test",
        "selected_C": selected_C,
        "rows": len(final_test_df),
        "positives": int(y_final.sum()),
        "positive_rate": float(y_final.mean()),
        "mean_predicted_probability": float(np.mean(final_prob)),
    }
)

prediction_frames.append(
    pd.DataFrame(
    {
        "deal_number": final_test_df["deal_number"].to_numpy(),
        "permno": final_test_df["permno"].to_numpy(),
        "year": final_test_df["year"].to_numpy(),
        "evaluation_stage": "final_test",
        "fold": "",
            "selected_C": selected_C,
            
        "outcome_dissident_favorable": y_final.to_numpy(),
        "outcome_mgmt_favorable": final_test_df["outcome_mgmt_favorable"].to_numpy(),
        "outcome_class": final_test_df["outcome_class"].to_numpy(),
        "pred_outcome_ridge_winsor": final_prob,
    }
)
)
# =========================================================
# 12. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "selected_C": selected_C,
            "pr_auc": selected_cv_metrics_df["pr_auc"].mean(),
            "roc_auc": selected_cv_metrics_df["roc_auc"].mean(),
            "brier_score": selected_cv_metrics_df["brier_score"].mean(),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "selected_C": selected_C,
            "pr_auc": selected_cv_metrics_df["pr_auc"].std(ddof=1),
            "roc_auc": selected_cv_metrics_df["roc_auc"].std(ddof=1),
            "brier_score": selected_cv_metrics_df["brier_score"].std(ddof=1),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        final_metrics,
    ]
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)

# =========================================================
# 13. EXTRACT COEFFICIENTS FROM FULL-DEVELOPMENT FIT
# =========================================================
preprocessor_fitted = final_pipeline.named_steps["preprocessor"]
model_fitted = final_pipeline.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()
coef_values = model_fitted.coef_.ravel()

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coef_values,
        "abs_coefficient": np.abs(coef_values),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

intercept_df = pd.DataFrame(
    {
        "feature": ["INTERCEPT"],
        "coefficient": [model_fitted.intercept_[0]],
        "abs_coefficient": [abs(model_fitted.intercept_[0])],
    }
)

coef_df = pd.concat([intercept_df, coef_df], ignore_index=True)

# =========================================================
# 14. SAVE OUTPUTS
# =========================================================
tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)
selected_cv_metrics_df.to_csv(CV_FOLD_METRICS_CSV, index=False)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
coef_df.to_csv(COEFFICIENTS_CSV, index=False)

# =========================================================
# 15. DISPLAY RESULTS
# =========================================================
print("Selected C:", selected_C)
print("\nSaved:")
print(TUNING_SUMMARY_CSV)
print(CV_FOLD_METRICS_CSV)
print(STAGE_METRICS_CSV)
print(PREDICTIONS_CSV)
print(COEFFICIENTS_CSV)

print("\nTuning summary:")
print(tuning_summary_df.to_string(index=False))

print("\nSelected-C development CV fold metrics:")
print(selected_cv_metrics_df.to_string(index=False))

print("\nStage metrics:")
print(stage_metrics_df.to_string(index=False))

print("\nTop 20 coefficients by absolute magnitude:")
print(coef_df.head(20).to_string(index=False))

Selected C: 10.0

Saved:
Outcome_Model_03_Ridge_Winsor_v1_Tuning_Summary.csv
Outcome_Model_03_Ridge_Winsor_v1_SelectedC_DevCV_Fold_Metrics.csv
Outcome_Model_03_Ridge_Winsor_v1_Stage_Metrics.csv
Outcome_Model_03_Ridge_Winsor_v1_Predictions.csv
Outcome_Model_03_Ridge_Winsor_v1_Coefficients.csv

Tuning summary:
     C  mean_pr_auc  std_pr_auc  mean_roc_auc  std_roc_auc  mean_brier_score  std_brier_score
 10.00     0.238740    0.088957      0.467225     0.056474          0.186706         0.046333
100.00     0.237941    0.084788      0.466108     0.051169          0.188482         0.046748
  0.10     0.230716    0.074542      0.480603     0.084442          0.179107         0.039644
  0.01     0.229894    0.059928      0.505432     0.089600          0.178828         0.028609
  1.00     0.228079    0.080580      0.460129     0.058597          0.182436         0.043602

Selected-C development CV fold metrics:
  pr_auc  roc_auc  brier_score  selected_C   fold  train_year_start  train_year_end  

In [6]:
# Outcome Model 04 v1: Elastic-net logistic regression, with winsorization
# C and l1_ratio tuned by six-fold expanding-window CV through 2021
# Thesis-standard implementation under final outcome v1 split protocol
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Outcome_Master_Modeling_Panel_v1.csv"

TUNING_SUMMARY_CSV = BASE_DIR / "Outcome_Model_04_ElasticNet_Winsor_v1_Tuning_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Outcome_Model_04_ElasticNet_Winsor_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Outcome_Model_04_ElasticNet_Winsor_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Outcome_Model_04_ElasticNet_Winsor_v1_Predictions.csv"
COEFFICIENTS_CSV = BASE_DIR / "Outcome_Model_04_ElasticNet_Winsor_v1_Coefficients.csv"

# Candidate hyperparameter grids
C_GRID = [0.01, 0.1, 1.0, 10.0]
L1_RATIO_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED OUTCOME TARGET + SPLITS
# =========================================================
TARGET_COL = "outcome_dissident_favorable"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()


# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED OUTCOME SPEC)
# =========================================================
# Notes:
# - The outcome extension inherits the locked 73 firm-side predictors from the
#   core thesis and adds the 11 campaign-side predictors locked in the
#   outcome specification workbook / whitelist.
# - The locked raw spec uses ff17_code as the coarse industry control.
#   For practical modeling we materialize this as a categorical feature
#   named ff17_for_model, with "Other" omitted as the reference category
#   and missing ff17 classifications mapped to "Unclassified".
# - Raw count columns retained in the panel for audit purposes are NOT used
#   directly in the model. The modeled count predictors are the log1p
#   transformed versions that were explicitly locked in the outcome panel.

continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
    "prior_campaign_same_target_past_365d_count_log1p",
    "activist_prior_campaigns_10y_raw_log1p",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "demand_group_board_governance",
    "demand_group_ma_strategic_alternatives",
    "demand_group_operational_strategy",
    "demand_group_control_hostile",
    "multi_demand_flag",
    "coalition_campaign",
    "universal_proxy_regime",
    "prior_campaign_same_target_past_365d",
    "activist_has_prior_campaign_10y_raw",
]

# Operationalized categorical industry control for the locked ff17_code spec
categorical_feature = "ff17_for_model"

# Build ff17 categorical field from the locked ff17_code mapping
ff17_code_to_short = {
    1: "Food",
    2: "Mines",
    3: "Oil",
    4: "Clths",
    5: "Durbl",
    6: "Chems",
    7: "Cnsum",
    8: "Cnstr",
    9: "Steel",
    10: "FabPr",
    11: "Machn",
    12: "Cars",
    13: "Trans",
    14: "Utils",
    15: "Rtail",
    16: "Finan",
    17: "Other",
}
df[categorical_feature] = (
    pd.to_numeric(df["ff17_code"], errors="coerce")
    .round()
    .astype("Int64")
    .map(ff17_code_to_short)
    .fillna("Unclassified")
    .astype(str)
)

# Fixed category order for auditability
ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

# Locked predictor whitelist cross-check
WHITELIST_PATH = BASE_DIR / "Outcome_Model_Predictor_Whitelist_v1.csv"
whitelist_df = pd.read_csv(WHITELIST_PATH)
locked_raw_predictors = whitelist_df["predictor"].tolist()

raw_modeled_predictors = continuous_features + binary_features + ["ff17_code"]

missing_in_code = sorted(set(locked_raw_predictors) - set(raw_modeled_predictors))
extra_in_code = sorted(set(raw_modeled_predictors) - set(locked_raw_predictors))
if missing_in_code or extra_in_code:
    raise ValueError(
        "Predictor whitelist mismatch. "
        f"Missing in code: {missing_in_code}. Extra in code: {extra_in_code}."
    )

# Basic safety check
required_cols = raw_modeled_predictors + [TARGET_COL, "year", "permno", "deal_number"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSING + MODEL BUILDER
# =========================================================
def build_pipeline(C_value: float, l1_ratio_value: float) -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[
            ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    elastic_net_model = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=C_value,
        l1_ratio=l1_ratio_value,
        max_iter=10000,
        random_state=42,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", elastic_net_model),
        ]
    )

# =========================================================
# 8. OPTION B: DEVELOPMENT CV FOLDS
# =========================================================
# Expanding-window development CV under the final v2 protocol
# Fold 1: train 2010-2015 -> validate 2016
# Fold 2: train 2010-2016 -> validate 2017
# Fold 3: train 2010-2017 -> validate 2018
# Fold 4: train 2010-2018 -> validate 2019
# Fold 5: train 2010-2019 -> validate 2020
# Fold 6: train 2010-2020 -> validate 2021

cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]


# =========================================================
# 9. TUNE (C, l1_ratio) BY DEVELOPMENT CV
# =========================================================
tuning_rows = []

for C_value in C_GRID:
    for l1_ratio_value in L1_RATIO_GRID:
        for fold_def in cv_fold_defs:
            fold_name = fold_def["fold"]
            train_years = fold_def["train_years"]
            val_years = fold_def["val_years"]

            train_df = df[df["year"].isin(train_years)].copy()
            val_df = df[df["year"].isin(val_years)].copy()

            X_train = train_df[all_predictors].copy()
            y_train = train_df[TARGET_COL].copy()

            X_val = val_df[all_predictors].copy()
            y_val = val_df[TARGET_COL].copy()

            pipeline = build_pipeline(C_value, l1_ratio_value)
            pipeline.fit(X_train, y_train)
            val_prob = pipeline.predict_proba(X_val)[:, 1]

            fold_metrics = evaluate_predictions(y_val, val_prob)
            fold_metrics.update(
                {
                    "C": C_value,
                    "l1_ratio": l1_ratio_value,
                    "fold": fold_name,
                    "train_year_start": min(train_years),
                    "train_year_end": max(train_years),
                    "val_year_start": min(val_years),
                    "val_year_end": max(val_years),
                    "train_rows": len(train_df),
                    "train_positives": int(y_train.sum()),
                    "train_positive_rate": float(y_train.mean()),
                    "val_rows": len(val_df),
                    "val_positives": int(y_val.sum()),
                    "val_positive_rate": float(y_val.mean()),
                    "mean_predicted_probability": float(np.mean(val_prob)),
                }
            )
            tuning_rows.append(fold_metrics)

tuning_detail_df = pd.DataFrame(tuning_rows)

tuning_summary_df = (
    tuning_detail_df
    .groupby(["C", "l1_ratio"], as_index=False)
    .agg(
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        mean_brier_score=("brier_score", "mean"),
        std_brier_score=("brier_score", "std"),
    )
)

# Selection rule:
# 1. highest mean PR-AUC
# 2. if tie, highest mean ROC-AUC
# 3. if tie, smaller C (more conservative)
# 4. if tie, smaller l1_ratio (more stable / less sparse)
tuning_summary_df = tuning_summary_df.sort_values(
    by=["mean_pr_auc", "mean_roc_auc", "C", "l1_ratio"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

selected_C = float(tuning_summary_df.loc[0, "C"])
selected_l1_ratio = float(tuning_summary_df.loc[0, "l1_ratio"])

# =========================================================
# 10. RERUN SELECTED HP FOR FOLD METRICS + OOS PREDICTIONS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(selected_C, selected_l1_ratio)
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "selected_C": selected_C,
            "selected_l1_ratio": selected_l1_ratio,
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "deal_number": val_df["deal_number"].to_numpy(),
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "selected_C": selected_C,
                "selected_l1_ratio": selected_l1_ratio,
                
                "outcome_dissident_favorable": y_val.to_numpy(),
                "outcome_mgmt_favorable": val_df["outcome_mgmt_favorable"].to_numpy(),
                "outcome_class": val_df["outcome_class"].to_numpy(),
                "pred_outcome_elasticnet_winsor": val_prob,
            }
        )
    )
selected_cv_metrics_df = pd.DataFrame(selected_cv_rows)

# =========================================================
# 11. FIT SELECTED HP ON FULL DEVELOPMENT, EVALUATE FINAL TEST
# =========================================================
development_df = df[df["split"] == "development"].copy()
final_test_df = df[df["split"] == "final_test"].copy()

X_dev = development_df[all_predictors].copy()
y_dev = development_df[TARGET_COL].copy()

final_pipeline = build_pipeline(selected_C, selected_l1_ratio)
final_pipeline.fit(X_dev, y_dev)

# Final test
X_final = final_test_df[all_predictors].copy()
y_final = final_test_df[TARGET_COL].copy()
final_prob = final_pipeline.predict_proba(X_final)[:, 1]

final_metrics = evaluate_predictions(y_final, final_prob)
final_metrics.update(
    {
        "stage": "final_test",
        "selected_C": selected_C,
        "selected_l1_ratio": selected_l1_ratio,
        "rows": len(final_test_df),
        "positives": int(y_final.sum()),
        "positive_rate": float(y_final.mean()),
        "mean_predicted_probability": float(np.mean(final_prob)),
    }
)

prediction_frames.append(
    pd.DataFrame(
    {
        "deal_number": final_test_df["deal_number"].to_numpy(),
        "permno": final_test_df["permno"].to_numpy(),
        "year": final_test_df["year"].to_numpy(),
        "evaluation_stage": "final_test",
        "fold": "",
            "selected_C": selected_C,
            "selected_l1_ratio": selected_l1_ratio,
            
        "outcome_dissident_favorable": y_final.to_numpy(),
        "outcome_mgmt_favorable": final_test_df["outcome_mgmt_favorable"].to_numpy(),
        "outcome_class": final_test_df["outcome_class"].to_numpy(),
        "pred_outcome_elasticnet_winsor": final_prob,
    }
)
)
# =========================================================
# 12. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "selected_C": selected_C,
            "selected_l1_ratio": selected_l1_ratio,
            "pr_auc": selected_cv_metrics_df["pr_auc"].mean(),
            "roc_auc": selected_cv_metrics_df["roc_auc"].mean(),
            "brier_score": selected_cv_metrics_df["brier_score"].mean(),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "selected_C": selected_C,
            "selected_l1_ratio": selected_l1_ratio,
            "pr_auc": selected_cv_metrics_df["pr_auc"].std(ddof=1),
            "roc_auc": selected_cv_metrics_df["roc_auc"].std(ddof=1),
            "brier_score": selected_cv_metrics_df["brier_score"].std(ddof=1),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        final_metrics,
    ]
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)

# =========================================================
# 13. EXTRACT COEFFICIENTS FROM FULL-DEVELOPMENT FIT
# =========================================================
preprocessor_fitted = final_pipeline.named_steps["preprocessor"]
model_fitted = final_pipeline.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()
coef_values = model_fitted.coef_.ravel()

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coef_values,
        "abs_coefficient": np.abs(coef_values),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

intercept_df = pd.DataFrame(
    {
        "feature": ["INTERCEPT"],
        "coefficient": [model_fitted.intercept_[0]],
        "abs_coefficient": [abs(model_fitted.intercept_[0])],
    }
)

coef_df = pd.concat([intercept_df, coef_df], ignore_index=True)

# =========================================================
# 14. SAVE OUTPUTS
# =========================================================
tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)
selected_cv_metrics_df.to_csv(CV_FOLD_METRICS_CSV, index=False)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
coef_df.to_csv(COEFFICIENTS_CSV, index=False)

# =========================================================
# 15. DISPLAY RESULTS
# =========================================================
print("Selected C:", selected_C)
print("Selected l1_ratio:", selected_l1_ratio)
print("\nSaved:")
print(TUNING_SUMMARY_CSV)
print(CV_FOLD_METRICS_CSV)
print(STAGE_METRICS_CSV)
print(PREDICTIONS_CSV)
print(COEFFICIENTS_CSV)

print("\nTuning summary:")
print(tuning_summary_df.to_string(index=False))

print("\nSelected-hyperparameter development CV fold metrics:")
print(selected_cv_metrics_df.to_string(index=False))

print("\nStage metrics:")
print(stage_metrics_df.to_string(index=False))

print("\nTop 20 coefficients by absolute magnitude:")
print(coef_df.head(20).to_string(index=False))

Selected C: 0.01
Selected l1_ratio: 0.3

Saved:
Outcome_Model_04_ElasticNet_Winsor_v1_Tuning_Summary.csv
Outcome_Model_04_ElasticNet_Winsor_v1_SelectedHP_DevCV_Fold_Metrics.csv
Outcome_Model_04_ElasticNet_Winsor_v1_Stage_Metrics.csv
Outcome_Model_04_ElasticNet_Winsor_v1_Predictions.csv
Outcome_Model_04_ElasticNet_Winsor_v1_Coefficients.csv

Tuning summary:
    C  l1_ratio  mean_pr_auc  std_pr_auc  mean_roc_auc  std_roc_auc  mean_brier_score  std_brier_score
 0.01       0.3     0.251657    0.074791      0.504353     0.118588          0.183703         0.016576
 0.01       0.1     0.247638    0.072447      0.512421     0.097564          0.178042         0.024960
 0.10       0.7     0.241722    0.061216      0.489169     0.113387          0.174078         0.038301
10.00       0.7     0.238614    0.090385      0.465781     0.055556          0.186631         0.045788
10.00       0.9     0.238361    0.090652      0.465578     0.055538          0.186683         0.045678
10.00       0.3     0.2

In [7]:
print("Loaded panel path:", PANEL_PATH)
print("Target columns present:", [c for c in df.columns if "outcome" in c.lower()])

assert "outcome_dissident_favorable" in df.columns, "Missing outcome_dissident_favorable in panel."
assert "outcome_mgmt_favorable" in df.columns, "Missing outcome_mgmt_favorable in panel."

cross_tab = pd.crosstab(df["outcome_mgmt_favorable"], df["outcome_dissident_favorable"])
print(cross_tab)

assert ((df["outcome_dissident_favorable"] == 1 - df["outcome_mgmt_favorable"]).all()), \
    "Outcome columns are not perfect complements."

Loaded panel path: Outcome_Master_Modeling_Panel_v1.csv
Target columns present: ['outcome_mgmt_favorable', 'outcome_dissident_favorable', 'outcome_class']
outcome_dissident_favorable    0    1
outcome_mgmt_favorable               
0                              0  435
1                            966    0
